In [111]:
import os
import numpy as np
from scipy.io import loadmat
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ================= 配置 =================
rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
rcParams['axes.unicode_minus'] = False

DATA_DIR = r'.\\data\\WT-行星齿轮数据集'
FILE_MAPPING = {
    'B1_35.MAT': 'Broken',
    'N1_35.MAT': 'Healthy',
    'M1_35.MAT': 'Missing_tooth',
    'R1_35.MAT': 'Root_crack',
    'W1_35.MAT': 'Wear'
}
WINDOW_SIZE = 2048
STRIDE = 2048


# ================= 归一化工具函数（Z-score）=================
def standardize_signal(x: np.ndarray) -> tuple:
    """Z-score 标准化: (x - mean) / std"""
    x_mean = x.mean()
    x_std = x.std()
    eps = 1e-8
    if x_std < eps:
        x_std = eps
        standardized_x = np.zeros_like(x)
    else:
        standardized_x = (x - x_mean) / x_std
    return standardized_x, {'mean': x_mean, 'std': x_std}


def destandardize_signal(x_norm: np.ndarray, mean: float,
                         std: float) -> np.ndarray:
    """反标准化: x * std + mean"""
    return x_norm * std + mean


print("🔄 正在流式加载数据：先 Z-score 标准化整条信号，再切分窗口...")

X_segments = []
y_labels = []

# 保存每个文件的标准化参数（用于后续可能的反标准化）
file_stats = {}

for file_name, fault_type in FILE_MAPPING.items():
    file_path = os.path.join(DATA_DIR, file_name)
    if not os.path.exists(file_path):
        print(f"⚠️ 跳过缺失文件: {file_name}")
        continue

    try:
        # 加载 .mat 数据
        mat_data = loadmat(file_path)
        raw_data = None
        for k, v in mat_data.items():
            if not k.startswith('__') and isinstance(
                    v, np.ndarray) and v.ndim == 2 and v.shape[1] >= 1:
                raw_data = v[:, :1]  # 只取第1列（注意：原代码写的是[:1]，不是前两列）
                break
        if raw_data is None:
            print(f"❌ 跳过无效文件: {file_name}")
            continue

        n = len(raw_data)
        if n < WINDOW_SIZE:
            print(f"⚠️ {file_name} 数据太短 ({n} < {WINDOW_SIZE})，跳过")
            continue

        # === 先对整条信号做 Z-score 标准化 ===
        standardized_raw, stats = standardize_signal(raw_data)
        file_stats[file_name] = stats  # 保存 mean/std

        # === 再切分窗口 ===
        count = 0
        for i in range(0, n - WINDOW_SIZE + 1, STRIDE):
            segment = standardized_raw[i:i + WINDOW_SIZE]  # [L, 1]
            X_segments.append(segment.copy())
            y_labels.append(fault_type)
            count += 1

        print(f"✅ {file_name} ({fault_type}): 标准化后切分 {count} 个窗口")

    except Exception as e:
        print(f"❌ 处理 {file_name} 出错: {e}")

if not X_segments:
    raise RuntimeError("未生成任何有效样本！")

# 转为 NumPy 数组
X_all = np.stack(X_segments, axis=0)  # [N, L, C]
y_all = np.array(y_labels)

print(f"\n✅ 总样本数: {len(X_all)}, 形状: {X_all.shape}")

🔄 正在流式加载数据：先 Z-score 标准化整条信号，再切分窗口...
✅ B1_35.MAT (Broken): 标准化后切分 7064 个窗口
✅ N1_35.MAT (Healthy): 标准化后切分 7080 个窗口
✅ M1_35.MAT (Missing_tooth): 标准化后切分 7096 个窗口
✅ R1_35.MAT (Root_crack): 标准化后切分 7120 个窗口
✅ W1_35.MAT (Wear): 标准化后切分 7064 个窗口

✅ 总样本数: 35424, 形状: (35424, 2048, 1)


In [212]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# ================================
# 🎯 保留原始多类标签（不转二分类！）
# ================================

# 假设 y_all 是字符串标签，例如: ['Healthy', 'InnerRace', 'Ball', 'OuterRace', ...]
# X_all: [N, L, 1] 或 [N, L] 的 numpy array

# 构建类别映射（字符串 ↔ 整数）
unique_labels = np.unique(y_all)
label_to_int = {label: idx for idx, label in enumerate(unique_labels)}
int_to_label = {idx: label for label, idx in label_to_int.items()}
num_classes = len(unique_labels)

print(f"🏷️ 多分类标签映射 (共 {num_classes} 类):")
for label, idx in label_to_int.items():
    print(f"   '{label}' → {idx}")

# 将字符串标签转为整数标签
y_all_int = np.array([label_to_int[label] for label in y_all])

# 提取正常数据（用于预训练）—— 仍用原始字符串判断
healthy_mask = (y_all == 'Healthy')
X_healthy = X_all[healthy_mask]
print(f"🟢 正常样本数（用于预训练）: {len(X_healthy)}")

# ================================
# 🔀 划分数据集（按多类标签分层）
# ================================

X_temp, X_test, y_temp_int, y_test_int = train_test_split(
    X_all,
    y_all_int,
    test_size=0.2,
    random_state=42,
    stratify=y_all_int  # 👈 按多类分层，确保每类在各集合中都有
)

X_train, X_val, y_train_int, y_val_int = train_test_split(X_temp,
                                                          y_temp_int,
                                                          test_size=0.25,
                                                          random_state=42,
                                                          stratify=y_temp_int)

# ================================
# 📦 Dataset 定义（支持多分类）
# ================================


class PretrainDataset(Dataset):

    def __init__(self, data_np):
        # 假设输入是 [N, L, 1] 或 [N, L]
        if data_np.ndim == 2:
            data_np = data_np[:, :, np.newaxis]  # 转为 [N, L, 1]
        self.data = torch.FloatTensor(data_np).permute(0, 2, 1)  # [N, 1, L]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


class FinetuneDataset(Dataset):

    def __init__(self, data_np, labels_int):
        if data_np.ndim == 2:
            data_np = data_np[:, :, np.newaxis]
        self.data = torch.FloatTensor(data_np).permute(0, 2, 1)
        self.labels = torch.LongTensor(labels_int)  # 已是整数标签

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


# ================================
# 🚀 创建 DataLoader
# ================================

batch_size = 64

pretrain_loader = DataLoader(PretrainDataset(X_healthy),
                             batch_size=batch_size,
                             shuffle=True)

train_loader = DataLoader(FinetuneDataset(X_train, y_train_int),
                          batch_size=batch_size,
                          shuffle=True)

val_loader = DataLoader(FinetuneDataset(X_val, y_val_int),
                        batch_size=batch_size,
                        shuffle=False)

test_loader = DataLoader(FinetuneDataset(X_test, y_test_int),
                         batch_size=batch_size,
                         shuffle=False)

# ================================
# 📊 打印统计信息
# ================================

from collections import Counter


def count_labels(y_int, int_to_label):
    counts = Counter(y_int)
    return {int_to_label[idx]: cnt for idx, cnt in sorted(counts.items())}


print(f"\n✅ 多分类数据管道构建完成！(共 {num_classes} 类)")
print(
    f"   - 训练集: {len(y_train_int)} 样本 → {count_labels(y_train_int, int_to_label)}"
)
print(
    f"   - 验证集: {len(y_val_int)} 样本 → {count_labels(y_val_int, int_to_label)}")
print(
    f"   - 测试集: {len(y_test_int)} 样本 → {count_labels(y_test_int, int_to_label)}"
)

🏷️ 多分类标签映射 (共 5 类):
   'Broken' → 0
   'Healthy' → 1
   'Missing_tooth' → 2
   'Root_crack' → 3
   'Wear' → 4
🟢 正常样本数（用于预训练）: 7080

✅ 多分类数据管道构建完成！(共 5 类)
   - 训练集: 21254 样本 → {np.str_('Broken'): 4238, np.str_('Healthy'): 4248, np.str_('Missing_tooth'): 4258, np.str_('Root_crack'): 4272, np.str_('Wear'): 4238}
   - 验证集: 7085 样本 → {np.str_('Broken'): 1413, np.str_('Healthy'): 1416, np.str_('Missing_tooth'): 1419, np.str_('Root_crack'): 1424, np.str_('Wear'): 1413}
   - 测试集: 7085 样本 → {np.str_('Broken'): 1413, np.str_('Healthy'): 1416, np.str_('Missing_tooth'): 1419, np.str_('Root_crack'): 1424, np.str_('Wear'): 1413}


In [229]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# 假设你已有：
# X_all: [N, L] 或 [N, L, 1]
# y_all: 原始字符串标签，如 ['Healthy', 'InnerRace', 'Ball', ...]

# ================================
# 🎯 步骤 1: 按原始多类标签划分数据集（分层）
# ================================

unique_labels = np.unique(y_all)
label_to_int = {label: i for i, label in enumerate(unique_labels)}
y_all_int = np.array([label_to_int[y] for y in y_all])

X_temp, X_test, y_temp_str, y_test_str = train_test_split(X_all,
                                                          y_all,
                                                          test_size=0.2,
                                                          random_state=42,
                                                          stratify=y_all_int)

X_train, X_val, y_train_str, y_val_str = train_test_split(
    X_temp,
    y_temp_str,
    test_size=0.25,
    random_state=42,
    stratify=[label_to_int[y] for y in y_temp_str])

print("✅ 数据集划分完成（按原始故障类型分层）")

# ================================
# 🎯 步骤 2: 小样本采样函数（仅用于 train/val）
# ================================


def sample_fewshot_binary(X, y_str, n_per_class=100, random_seed=42):
    rng = np.random.RandomState(random_seed)
    unique = np.unique(y_str)
    X_list, y_bin_list, y_str_list = [], [], []

    for label in unique:
        mask = (y_str == label)
        X_label = X[mask]
        y_label = y_str[mask]
        n_avail = len(X_label)
        n_take = min(n_per_class, n_avail)

        if n_take < n_avail:
            idx = rng.choice(n_avail, n_take, replace=False)
            X_label = X_label[idx]
            y_label = y_label[idx]

        X_list.append(X_label)
        y_str_list.append(y_label)
        bin_val = 0 if label == 'Healthy' else 1
        y_bin_list.append(np.full(len(X_label), bin_val))

    X_out = np.concatenate(X_list, axis=0)
    y_bin_out = np.concatenate(y_bin_list, axis=0)
    y_str_out = np.concatenate(y_str_list, axis=0)

    shuffle_idx = rng.permutation(len(X_out))
    return X_out[shuffle_idx], y_bin_out[shuffle_idx], y_str_out[shuffle_idx]


# ================================
# 🚀 应用小样本采样（仅训练集和验证集）
# ================================

n_per_class = 100

X_train_fs, y_train_bin, y_train_str_fs = sample_fewshot_binary(
    X_train, y_train_str, n_per_class=n_per_class, random_seed=42)

X_val_fs, y_val_bin, y_val_str_fs = sample_fewshot_binary(
    X_val, y_val_str, n_per_class=n_per_class, random_seed=123)


# ✅ 测试集：不采样！保留全部样本，仅转为二分类
def to_binary_labels_str(y_str):
    return np.where(y_str == 'Healthy', 0, 1).astype(int)


X_test_fs = X_test  # 全量测试集
y_test_bin = to_binary_labels_str(y_test_str)
y_test_str_fs = y_test_str  # 用于分析

# ================================
# 📦 Dataset 和 DataLoader
# ================================


class FinetuneDataset(Dataset):

    def __init__(self, data_np, labels):
        if data_np.ndim == 2:
            data_np = data_np[:, :, np.newaxis]
        self.data = torch.FloatTensor(data_np).permute(0, 2, 1)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


batch_size = 64

train_loader = DataLoader(FinetuneDataset(X_train_fs, y_train_bin),
                          batch_size=batch_size,
                          shuffle=True)
val_loader = DataLoader(FinetuneDataset(X_val_fs, y_val_bin),
                        batch_size=batch_size,
                        shuffle=False)
test_loader = DataLoader(FinetuneDataset(X_test_fs, y_test_bin),
                         batch_size=batch_size,
                         shuffle=False)

# ================================
# 📊 打印统计信息
# ================================


def print_fault_distribution(name, y_str):
    counts = Counter(y_str)
    healthy = counts.get('Healthy', 0)
    faulty_total = sum(v for k, v in counts.items() if k != 'Healthy')
    fault_types = {k: v for k, v in counts.items() if k != 'Healthy'}
    print(f"\n{name} 集:")
    print(f"  总样本: {len(y_str)} | Healthy: {healthy}, Faulty: {faulty_total}")
    print(f"  故障子类分布: {dict(sorted(fault_types.items()))}")


print("\n" + "=" * 60)
print(f"✅ 小样本二分类数据集构建完成！(Train/Val 每原始类 ≤{n_per_class} 样本，Test 全量)")
print("=" * 60)
print_fault_distribution("训练", y_train_str_fs)
print_fault_distribution("验证", y_val_str_fs)
print_fault_distribution("测试", y_test_str_fs)  # 显示完整测试分布

print(f"\n📊 二分类统计:")
print(
    f"   - 训练集: {len(y_train_bin)} 样本 (Healthy: {(y_train_bin==0).sum()}, Faulty: {(y_train_bin==1).sum()})"
)
print(
    f"   - 验证集: {len(y_val_bin)} 样本 (Healthy: {(y_val_bin==0).sum()}, Faulty: {(y_val_bin==1).sum()})"
)
print(
    f"   - 测试集: {len(y_test_bin)} 样本 (Healthy: {(y_test_bin==0).sum()}, Faulty: {(y_test_bin==1).sum()}) [全量]"
)

✅ 数据集划分完成（按原始故障类型分层）

✅ 小样本二分类数据集构建完成！(Train/Val 每原始类 ≤100 样本，Test 全量)

训练 集:
  总样本: 500 | Healthy: 100, Faulty: 400
  故障子类分布: {np.str_('Broken'): 100, np.str_('Missing_tooth'): 100, np.str_('Root_crack'): 100, np.str_('Wear'): 100}

验证 集:
  总样本: 500 | Healthy: 100, Faulty: 400
  故障子类分布: {np.str_('Broken'): 100, np.str_('Missing_tooth'): 100, np.str_('Root_crack'): 100, np.str_('Wear'): 100}

测试 集:
  总样本: 7085 | Healthy: 1416, Faulty: 5669
  故障子类分布: {np.str_('Broken'): 1413, np.str_('Missing_tooth'): 1419, np.str_('Root_crack'): 1424, np.str_('Wear'): 1413}

📊 二分类统计:
   - 训练集: 500 样本 (Healthy: 100, Faulty: 400)
   - 验证集: 500 样本 (Healthy: 100, Faulty: 400)
   - 测试集: 7085 样本 (Healthy: 1416, Faulty: 5669) [全量]


In [215]:
sample_x = next(iter(val_loader))[1]
sample_x.shape

torch.Size([64])

In [216]:
sample_x

tensor([1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1])

In [ ]:
import torch
import torch.nn as nn
import math
from typing import Tuple
import random


class SignalReconstructionBlock(nn.Module):
    """
    单信号通道的 MAE 重构块（适用于振动、温度等一维时间序列）
    
    输入: [B, 1, L] —— 要求已在外部归一化（如 (x - mean)/std）
    输出（训练）: loss (标量)
    输出（推理）: 原始 padded 信号、重建信号、mask 时间点（连续区域）
    """

    def __init__(self,
                 signal_length: int,
                 patch_size: int = 64,
                 embed_dim: int = 128,
                 encoder_depth: int = 4,
                 decoder_depth: int = 2,
                 num_heads: int = 8,
                 mask_ratio: float = 0.75,
                 dropout: float = 0.1,
                 use_input_norm: bool = False,
                 # 新增：block mask 参数
                 min_block_len: int = 3,
                 max_block_len: int = 4,
                 min_blocks: int = 1,
                 max_blocks: int = 3):
        super().__init__()
        self.signal_length = signal_length
        self.patch_size = patch_size
        self.mask_ratio = mask_ratio
        self.num_patches = math.ceil(signal_length / patch_size)
        self.use_input_norm = use_input_norm

        # Block mask 超参
        self.min_block_len = min_block_len
        self.max_block_len = max_block_len
        self.min_blocks = min_blocks
        self.max_blocks = max_blocks

        # === Patch Embedding ===
        self.patch_embed = nn.Conv1d(in_channels=1,
                                     out_channels=embed_dim,
                                     kernel_size=patch_size,
                                     stride=patch_size,
                                     padding=0)
        self.norm = nn.LayerNorm(embed_dim)

        # === Positional Embedding ===
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches, embed_dim))
        nn.init.normal_(self.pos_embed, std=0.02)

        # === Encoder ===
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim,
                                                   nhead=num_heads,
                                                   dim_feedforward=embed_dim * 4,
                                                   dropout=dropout,
                                                   batch_first=True,
                                                   activation='gelu')
        self.encoder = nn.TransformerEncoder(encoder_layer,
                                             num_layers=encoder_depth)
        self.encoder_norm = nn.LayerNorm(embed_dim)

        # === Decoder ===
        self.decoder_embed = nn.Linear(embed_dim, embed_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.normal_(self.mask_token, std=0.02)

        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim,
                                                   nhead=num_heads,
                                                   dim_feedforward=embed_dim * 4,
                                                   dropout=dropout,
                                                   batch_first=True,
                                                   activation='gelu')
        self.decoder = nn.TransformerDecoder(decoder_layer,
                                             num_layers=decoder_depth)
        self.decoder_norm = nn.LayerNorm(embed_dim)

        # === Prediction Head ===
        self.pred_head = nn.Linear(embed_dim, patch_size)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.pred_head.weight)
        if self.pred_head.bias is not None:
            nn.init.constant_(self.pred_head.bias, 0)

    def _pad_signal(self, x: torch.Tensor) -> torch.Tensor:
        L = x.size(-1)
        if L % self.patch_size != 0:
            pad_len = self.patch_size - (L % self.patch_size)
            x = nn.functional.pad(x, (0, pad_len))
        return x

    def _generate_block_mask_ids(self, B, N):
        """
        生成 block-wise mask 的 ids_shuffle 和 ids_restore。
        保证每个样本的 len_keep 相同（基于 mask_ratio）。
        """
        len_keep = int(N * (1 - self.mask_ratio))  # 固定保留数量！
        noise = torch.rand(B, N, device=self.patch_embed.weight.device)  # 初始化噪声

        # 对每个样本独立生成 block mask
        for i in range(B):
            # 1. 初始化全为 False 的 mask
            mask = torch.zeros(N, dtype=torch.bool)

            # 2. 随机生成若干连续 block
            num_blocks = torch.randint(self.min_blocks, self.max_blocks + 1, ()).item()
            placed = 0
            attempts = 0
            max_attempts = 100

            while placed < num_blocks and attempts < max_attempts:
                block_len = torch.randint(self.min_block_len, min(self.max_block_len + 1, N), ()).item()
                start = torch.randint(0, N - block_len + 1, ()).item()

                # 如果该区域未被 mask，就 mask 它
                if not mask[start:start + block_len].any():
                    mask[start:start + block_len] = True
                    placed += 1
                attempts += 1

            # 3. 如果 mask 数量不足，随机补足；如果超了，随机取消
            current_masked = mask.sum().item()
            target_masked = N - len_keep

            if current_masked < target_masked:
                # 补 mask
                unmasked_indices = (~mask).nonzero(as_tuple=True)[0]
                to_mask = unmasked_indices[torch.randperm(len(unmasked_indices))[:target_masked - current_masked]]
                mask[to_mask] = True
            elif current_masked > target_masked:
                # 取消部分 mask
                masked_indices = mask.nonzero(as_tuple=True)[0]
                to_unmask = masked_indices[torch.randperm(len(masked_indices))[:current_masked - target_masked]]
                mask[to_unmask] = False

            # 4. 构造排序：visible 在前，masked 在后
            ids_shuffle = torch.argsort(mask.float())  # False (0) < True (1)，所以 visible 在前
            noise[i] = torch.randperm(N).float() / N  # 可选：加随机扰动避免顺序固定
            # 或者直接用：noise[i] = torch.where(mask, 1.0, 0.0) + torch.rand(N) * 1e-6

        # 最终统一排序（确保每个样本有相同 len_keep）
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        return ids_shuffle, ids_restore, len_keep

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, L = x.shape
        assert C == 1 and L == self.signal_length, f"Expected [B, 1, {self.signal_length}], got {x.shape}"

        x_padded = self._pad_signal(x)  # [B, 1, L_pad]
        patches = self.patch_embed(x_padded).transpose(1, 2)  # [B, N, D]
        patches = self.norm(patches) + self.pos_embed
        N = patches.size(1)

        # === 使用 block mask 替代随机 noise ===
        ids_shuffle, ids_restore, len_keep = self._generate_block_mask_ids(B, N)

        ids_keep = ids_shuffle[:, :len_keep]
        visible = torch.gather(patches,
                               dim=1,
                               index=ids_keep.unsqueeze(-1).expand(-1, -1, patches.size(-1)))
        encoded = self.encoder_norm(self.encoder(visible))

        dec_visible = self.decoder_embed(encoded)
        mask_tokens = self.mask_token.repeat(B, N - len_keep, 1)
        full_dec = torch.cat([dec_visible, mask_tokens], dim=1)

        full_dec = torch.gather(full_dec,
                                dim=1,
                                index=ids_restore.unsqueeze(-1).expand(-1, -1, full_dec.size(-1)))
        full_dec = full_dec + self.pos_embed

        decoded = decoded = self.decoder_norm(
            self.decoder(tgt=full_dec, memory=encoded))
        pred = self.pred_head(decoded)

        target = x_padded.unfold(2, self.patch_size, self.patch_size).squeeze(1)  # [B, N, P]

        mask = torch.ones(B, N, device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        loss = ((pred - target) ** 2).mean(dim=-1)
        loss = (loss * mask).sum() / (mask.sum() + 1e-8)
        return loss


    @torch.no_grad()
    def reconstruct(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, C, L = x.shape
        assert C == 1 and L == self.signal_length

        x_orig_unnorm = x.clone().squeeze(1)  # [B, L] —— 原始未归一化信号（用于最终输出）
        x_padded = self._pad_signal(x)        # [B, 1, L_pad]
        L_pad = x_padded.size(-1)

        patches = self.patch_embed(x_padded).transpose(1, 2)  # [B, N, D]
        patches = self.norm(patches) + self.pos_embed
        N = patches.size(1)

        # 生成 block mask
        ids_shuffle, ids_restore, len_keep = self._generate_block_mask_ids(B, N)

        # Encoder 只处理 visible patches
        ids_keep = ids_shuffle[:, :len_keep]
        visible = torch.gather(patches, dim=1,
                            index=ids_keep.unsqueeze(-1).expand(-1, -1, patches.size(-1)))
        encoded = self.encoder_norm(self.encoder(visible))

        # Decoder 重建全部 patches
        dec_visible = self.decoder_embed(encoded)
        mask_tokens = self.mask_token.repeat(B, N - len_keep, 1)
        full_dec = torch.cat([dec_visible, mask_tokens], dim=1)
        full_dec = torch.gather(full_dec, dim=1,
                                index=ids_restore.unsqueeze(-1).expand(-1, -1, full_dec.size(-1)))
        full_dec = full_dec + self.pos_embed

        decoded = self.decoder_norm(self.decoder(tgt=full_dec, memory=encoded))
        pred = self.pred_head(decoded)  # [B, N, P]

        # 重建信号（全量）
        x_recon_full = pred.reshape(B, N * self.patch_size)  # [B, L_pad]

        # 构建 mask_time: True 表示该时间点被 mask（需要重建）
        mask = torch.ones(B, N, device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)  # [B, N]
        mask_time = mask.unsqueeze(-1).repeat(1, 1, self.patch_size).reshape(
            B, L_pad).bool()  # [B, L_pad]

        # ✅ 关键修改：混合重建 —— masked 区域用重建值，unmasked 区域用原始 padded 值
        x_padded_flat = x_padded.squeeze(1)  # [B, L_pad]
        x_recon_hybrid = torch.where(mask_time, x_recon_full, x_padded_flat)

        # 最终输出：
        # - x_orig_unnorm: 原始输入（长度 = L，未 padding）
        # - x_recon_hybrid: 混合重建（长度 = L_pad，但你通常只关心前 L）
        # - mask_time[:, :L]: mask 掩码（对齐原始长度）

        return x_orig_unnorm, x_recon_hybrid[:, :L], mask_time[:, :L]

    # ... [save_pretrained, from_pretrained, print_summary 等保持不变] ...

    def get_config(self) -> dict:
        cfg = {
            "signal_length": self.signal_length,
            "patch_size": self.patch_size,
            "embed_dim": self.patch_embed.out_channels,
            "encoder_depth": len(self.encoder.layers),
            "decoder_depth": len(self.decoder.layers),
            "num_heads": self.encoder.layers[0].self_attn.num_heads,
            "mask_ratio": self.mask_ratio,
            "use_input_norm": self.use_input_norm,
        }
        # 添加 block mask 配置
        cfg.update({
            "min_block_len": self.min_block_len,
            "max_block_len": self.max_block_len,
            "min_blocks": self.min_blocks,
            "max_blocks": self.max_blocks,
        })
        return cfg
    @torch.no_grad()
    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """
        提取编码器输出的全局特征（用于下游任务，如分类、聚类、异常检测）。
        
        Args:
            x: [B, 1, L] 输入信号（已归一化）
        
        Returns:
            features: [B, embed_dim] 全局特征向量
        """
        B, C, L = x.shape
        assert C == 1 and L == self.signal_length, f"Expected [B, 1, {self.signal_length}], got {x.shape}"

        x_padded = self._pad_signal(x)  # [B, 1, L_pad]
        patches = self.patch_embed(x_padded).transpose(1, 2)  # [B, N, D]
        patches = self.norm(patches) + self.pos_embed

        # 使用全部 patches（无 mask）通过 encoder
        encoded = self.encoder_norm(self.encoder(patches))  # [B, N, D]

        # 全局平均池化（也可尝试 max pooling 或 cls token）
        global_feat = encoded.mean(dim=1)  # [B, D]
        
        return global_feat

In [257]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------
# 配置（单通道）
# ----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "./checkpoints_mae"
VIS_DIR = "./mae_reconstructions"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

CHANNEL_IDX = 0
SIGNAL_LENGTH = X_healthy.shape[1]

MODEL_CFG = dict(
    signal_length=SIGNAL_LENGTH,
    patch_size=64,
    embed_dim=128,
    encoder_depth=4,
    decoder_depth=2,
    num_heads=8,
    mask_ratio=0.2,
    dropout=0.1,
    min_block_len=5,
    max_block_len=6,
    min_blocks=5,
    max_blocks=6,
)

# 训练参数
EPOCHS = 1000
LR = 1e-4
WEIGHT_DECAY = 0.05
SAVE_EVERY = 75
PATIENCE = 40
EARLY_STOPPING_DELTA = 1e-5

print(f"✅ 使用通道 {CHANNEL_IDX}")
print(f"📊 信号长度: {SIGNAL_LENGTH}")
print(f"📁 Checkpoint: {CHECKPOINT_DIR}")
print(f"🖼️  可视化: {VIS_DIR}")

# ----------------------------
# 初始化模型与优化器
# ----------------------------
model = SignalReconstructionBlock(**MODEL_CFG).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

if hasattr(model, 'print_summary'):
    model.print_summary()

# ----------------------------
# 🚀 简洁加载：仅加载权重，重置所有训练状态
# ----------------------------
CKPT_PATH = os.path.join(CHECKPOINT_DIR, f"mae_ch{CHANNEL_IDX}_best.pth")

# 初始训练状态（全新开始！）
start_epoch = 0
best_loss = float('inf')
epochs_no_improve = 0
early_stop = False

# 尝试加载预训练权重（仅用于初始化模型）
if os.path.exists(CKPT_PATH):
    print(f"🔄 加载预训练权重: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    print("   → 模型权重已加载，训练状态已重置（全新早停）")
else:
    print("🆕 未找到预训练模型，从头开始训练。")

# ----------------------------
# 训练循环
# ----------------------------
train_start_time = datetime.now()
for epoch in range(start_epoch, EPOCHS):
    if early_stop:
        print("🛑 触发早停！训练结束。")
        break

    model.train()
    total_loss = 0.0
    num_batches = 0

    pbar = tqdm(pretrain_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for x in pbar:
        x = x.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast('cuda'):
                loss = model(x)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = model(x)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix(Loss=f"{loss.item():.4f}")

    avg_loss = total_loss / num_batches

    # ----------------------------
    # 早停 & 保存最佳模型
    # ----------------------------
    is_best = False
    if avg_loss < best_loss - EARLY_STOPPING_DELTA:
        best_loss = avg_loss
        epochs_no_improve = 0
        is_best = True
    else:
        epochs_no_improve += 1

    # --- 🎯 关键修改：使用你想要的紧凑日志格式 ---
    elapsed = datetime.now() - train_start_time
    total_sec = int(elapsed.total_seconds())
    time_str = f"{total_sec // 60}m {total_sec % 60}s"

    # 主日志（两行）
    print(f"Train_Loss: {avg_loss:.6f} | Best_Loss:  {best_loss:.6f}")
    print(
        f"Time: {time_str:<9} | No_Improve: {epochs_no_improve} | Epoch {epoch+1}/{EPOCHS}"
    )

    # 如果是新最佳，额外提示
    if is_best:
        torch.save({'model': model.state_dict()}, CKPT_PATH)
        print(f"   ⭐ 发现更佳模型！已保存 (Loss: {best_loss:.6f})")

    # 早停检查
    if epochs_no_improve >= PATIENCE:
        early_stop = True
        print(f"\n🛑 早停触发！最终最佳 Loss: {best_loss:.6f}")
    # ----------------------------
    # 可视化重建（用 axvspan 画 mask 区域为透明色块）
    # ----------------------------
    if (epoch + 1) % SAVE_EVERY == 0 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            sample_x = next(iter(pretrain_loader))[0:1].to(DEVICE)  # [1, 1, T]
            x_orig, x_recon, mask_time = model.reconstruct(sample_x)

            x_orig = x_orig.cpu().numpy().flatten()  # [T]
            x_recon = x_recon.cpu().numpy().flatten()  # [T]
            mask_time = mask_time.cpu().numpy().flatten()  # [T], bool

            plt.figure(figsize=(12, 4))
            plt.plot(x_orig, label='Original', alpha=0.7, linewidth=1)
            plt.plot(x_recon, label='Reconstructed', alpha=0.9, linewidth=1)

            # 直接使用 mask_time（已是时间维度的布尔掩码）
            if mask_time.any():
                padded = np.concatenate([[False], mask_time, [False]])
                diff = np.diff(padded.astype(int))
                starts = np.where(diff == 1)[0]
                ends = np.where(diff == -1)[0]

                for i, (start, end) in enumerate(zip(starts, ends)):
                    plt.axvspan(start,
                                end,
                                color='lightcoral',
                                alpha=0.4,
                                label='Masked Region' if i == 0 else "")

            plt.title(
                f'Channel {CHANNEL_IDX} | Epoch {epoch+1} | Loss: {avg_loss:.4f}'
            )
            plt.xlabel('Time Step')
            plt.ylabel('Amplitude')
            plt.legend()
            plt.tight_layout()

            suffix = "_best" if is_best else ""
            vis_path = os.path.join(VIS_DIR,
                                    f"epoch_{epoch+1:03d}{suffix}.png")
            plt.savefig(vis_path, dpi=150)
            plt.close()
            print(f"   🖼️  重建图已保存: {vis_path}")

# ----------------------------
# 最终提示
# ----------------------------
print("\n🎉 单通道 MAE 训练完成！")
print(f"✅ 最佳模型: {CKPT_PATH}")
print(f"📊 最佳损失: {best_loss:.6f}")
print(f"🖼️  重建可视化: {VIS_DIR}/")

✅ 使用通道 0
📊 信号长度: 2048
📁 Checkpoint: ./checkpoints_mae
🖼️  可视化: ./mae_reconstructions
🔄 加载预训练权重: ./checkpoints_mae\mae_ch0_best.pth
   → 模型权重已加载，训练状态已重置（全新早停）


Train_Loss: 0.256791 | Best_Loss:  0.256791
Time: 0m 9s     | No_Improve: 0 | Epoch 1/1000
   ⭐ 发现更佳模型！已保存 (Loss: 0.256791)


Train_Loss: 0.254036 | Best_Loss:  0.254036
Time: 0m 19s    | No_Improve: 0 | Epoch 2/1000
   ⭐ 发现更佳模型！已保存 (Loss: 0.254036)


Train_Loss: 0.253315 | Best_Loss:  0.253315
Time: 0m 29s    | No_Improve: 0 | Epoch 3/1000
   ⭐ 发现更佳模型！已保存 (Loss: 0.253315)


Train_Loss: 0.254108 | Best_Loss:  0.253315
Time: 0m 39s    | No_Improve: 1 | Epoch 4/1000


Train_Loss: 0.254617 | Best_Loss:  0.253315
Time: 0m 49s    | No_Improve: 2 | Epoch 5/1000


KeyboardInterrupt: 

In [261]:
import torch
import torch.nn as nn
from collections import OrderedDict


class MAEClassifier(nn.Module):

    def __init__(self,
                 encoder: SignalReconstructionBlock,
                 num_classes: int=2,
                 freeze_encoder: bool = True):
        super().__init__()
        self.encoder = encoder
        self.num_classes = num_classes

        # 冻结编码器（Linear Probing）或允许微调（Full Fine-tuning）
        for param in self.encoder.parameters():
            param.requires_grad = not freeze_encoder

        # 分类头（接在全局特征后）
        self.classifier = nn.Sequential(
            nn.Linear(encoder.patch_embed.out_channels, 512), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(512, num_classes))

    def forward(self, x):
        # x: [B, 1, L]
        features = self.encoder.extract_features(x)  # [B, D]
        logits = self.classifier(features)  # [B, num_classes]
        return logits
# ========== 配置 ==========
num_epochs = 100
freeze_encoder = True  # 👈 先定义！True 表示 Linear Probing，False 表示 Full Fine-tuning
lr = 1e-3 if freeze_encoder else 5e-5  # 微调时学习率要小
weight_decay = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 假设 model 是你已加载的预训练 SignalReconstructionBlock 实例
# 如果还没加载，请确保下面这行有定义：
SIGNAL_LENGTH = X_healthy.shape[1]

MODEL_CFG = dict(
    signal_length=SIGNAL_LENGTH,
    patch_size=64,
    embed_dim=128,
    encoder_depth=4,
    decoder_depth=2,
    num_heads=8,
    mask_ratio=0.5,
    dropout=0.1,
    min_block_len=5,
    max_block_len=6,
    min_blocks=5,
    max_blocks=6,
)
# 正确加载预训练权重
checkpoint = torch.load("checkpoints_mae/mae_ch0_best.pth",
                        map_location=device)
# 或者用反斜杠（但建议用正斜杠或 os.path.join）
# checkpoint = torch.load(r"checkpoints_mae\mae_ch0_best.pth", map_location=device)

# 提取模型权重（关键！）
state_dict = checkpoint["model"]  # 👈 假设保存时用了 'model' 作为键

# 加载到你的模型
model.load_state_dict(state_dict, strict=True)



classifier_model = MAEClassifier(encoder=model,

                                 freeze_encoder=freeze_encoder).to(device)

optimizer = torch.optim.AdamW(classifier_model.parameters(),
                              lr=lr,
                              weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                       T_max=num_epochs)


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


num_trainable = count_trainable_params(classifier_model)
print(f"🔍 实验 A（预训练+冻结）可训练参数量: {num_trainable:,} (约 {num_trainable/1e3:.1f}K)")

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(torch.cat(all_labels), torch.cat(all_preds))
    return avg_loss, acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(torch.cat(all_labels), torch.cat(all_preds))
    return avg_loss, acc


# ========== 早停 & 内存缓存配置 ==========
PATIENCE = 20
EARLY_STOP_DELTA = 1e-4
best_val_acc = 0.0
epochs_no_improve = 0
early_stop = False
best_model_state = None  # 👈 用于在内存中保存最佳模型参数

for epoch in range(num_epochs):
    if early_stop:
        print("🛑 触发早停！训练提前结束。")
        break

    train_loss, train_acc = train_epoch(classifier_model, train_loader,
                                        criterion, optimizer, device)
    val_loss, val_acc = validate(classifier_model, val_loader, criterion,
                                 device)
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    # --- 🛑 早停 + 内存缓存最佳模型 ---
    if val_acc > best_val_acc + EARLY_STOP_DELTA:
        best_val_acc = val_acc
        epochs_no_improve = 0
        # 💾 不保存到磁盘，而是深拷贝 state_dict 到内存
        best_model_state = {
            k: v.cpu()
            for k, v in classifier_model.state_dict().items()
        }  # 转为 CPU 张量以节省 GPU 显存
        print(f"🎉 New best model cached in memory (Val Acc: {val_acc:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            early_stop = True
            print(f"⚠️ 验证准确率连续 {PATIENCE} 轮未提升，准备早停...")

print(f"\n✅ 训练完成！最佳验证准确率: {best_val_acc:.4f}")

# =============================================
# 🔍 最终验证：加载内存中的最佳模型状态
# =============================================
if best_model_state is not None:
    # 恢复最佳权重（自动适配当前设备）
    classifier_model.load_state_dict(best_model_state)
else:
    print("⚠️ 未找到有效最佳模型，使用最终模型进行评估。")

# 在验证集上做最终评估
final_test_loss, final_test_acc = validate(classifier_model, test_loader,
                                           criterion, device)

print(f"\n🔍 最终验证结果:")
print(f"   - 测试损失 (Loss): {final_test_loss:.4f}")
print(f"   - 测试准确率 (Accuracy): {final_test_acc:.4f}")

🔍 实验 A（预训练+冻结）可训练参数量: 67,074 (约 67.1K)
Epoch 1/100 | Train Loss: 0.6022, Acc: 0.8000 | Val Loss: 0.5347, Acc: 0.8000
🎉 New best model cached in memory (Val Acc: 0.8000)
Epoch 2/100 | Train Loss: 0.5026, Acc: 0.8000 | Val Loss: 0.4730, Acc: 0.8000
Epoch 3/100 | Train Loss: 0.4719, Acc: 0.8000 | Val Loss: 0.4579, Acc: 0.8000
Epoch 4/100 | Train Loss: 0.4403, Acc: 0.8000 | Val Loss: 0.4380, Acc: 0.8000
Epoch 5/100 | Train Loss: 0.4213, Acc: 0.8000 | Val Loss: 0.4241, Acc: 0.8020
🎉 New best model cached in memory (Val Acc: 0.8020)
Epoch 6/100 | Train Loss: 0.3998, Acc: 0.8120 | Val Loss: 0.4110, Acc: 0.8160
🎉 New best model cached in memory (Val Acc: 0.8160)
Epoch 7/100 | Train Loss: 0.3811, Acc: 0.8300 | Val Loss: 0.3962, Acc: 0.8200
🎉 New best model cached in memory (Val Acc: 0.8200)
Epoch 8/100 | Train Loss: 0.3624, Acc: 0.8400 | Val Loss: 0.3809, Acc: 0.8340
🎉 New best model cached in memory (Val Acc: 0.8340)
Epoch 9/100 | Train Loss: 0.3394, Acc: 0.8500 | Val Loss: 0.3665, Acc: 0.8440

In [260]:
# ==============================
# 实验 B：无预训练（随机初始化）
# ==============================

from re import T


print("\n" + "=" * 60)
print("🧪 对照实验：随机初始化编码器（无预训练）")
print("=" * 60)

# 1. 创建全新模型（不加载任何预训练权重）
random_encoder = SignalReconstructionBlock(**MODEL_CFG)  # 随机初始化
classifier_random = MAEClassifier(
    encoder=random_encoder,
    freeze_encoder=False # 必须设为 False，否则无法训练
).to(device)

# 2. 优化器 & 学习率调度
lr_for_random = 1e-4
optimizer_for_random = torch.optim.AdamW(classifier_random.parameters(),
                                         lr=lr_for_random,
                                         weight_decay=weight_decay)
scheduler_for_random = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_for_random, T_max=num_epochs)

# 3. 早停 & 内存缓存
best_val_acc_random = 0.0
epochs_no_improve_random = 0
early_stop_flag_random = False
best_state_dict_random = None


# 打印可训练参数数量
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


num_trainable = count_trainable_params(classifier_random)
print(f"🔍 实验 B（随机+冻结）可训练参数量: {num_trainable:,} (约 {num_trainable/1e3:.1f}K)")
# 4. 训练循环
for epoch_idx in range(num_epochs):
    if early_stop_flag_random:
        print("🛑 随机模型：触发早停！训练提前结束。")
        break

    train_loss_rand, train_acc_rand = train_epoch(classifier_random,
                                                  train_loader, criterion,
                                                  optimizer_for_random, device)
    val_loss_rand, val_acc_rand = validate(classifier_random, val_loader,
                                           criterion, device)
    scheduler_for_random.step()

    print(f"[Random] Epoch {epoch_idx+1}/{num_epochs} | "
          f"Train Loss: {train_loss_rand:.4f}, Acc: {train_acc_rand:.4f} | "
          f"Val Loss: {val_loss_rand:.4f}, Acc: {val_acc_rand:.4f}")

    if val_acc_rand > best_val_acc_random + EARLY_STOP_DELTA:
        best_val_acc_random = val_acc_rand
        epochs_no_improve_random = 0
        best_state_dict_random = {
            key: tensor.cpu()
            for key, tensor in classifier_random.state_dict().items()
        }
        print(
            f"🎉 [Random] New best model cached in memory (Val Acc: {val_acc_rand:.4f})"
        )
    else:
        epochs_no_improve_random += 1
        if epochs_no_improve_random >= PATIENCE:
            early_stop_flag_random = True
            print(f"⚠️ [Random] 连续 {PATIENCE} 轮未提升，早停...")

print(f"\n✅ 随机初始化模型训练完成！最佳验证准确率: {best_val_acc_random:.4f}")

# =============================================
# 🔍 最终在测试集上评估（使用实验 B 的模型）
# =============================================
if best_state_dict_random is not None:
    classifier_random.load_state_dict(best_state_dict_random)

# 注意：这里用的是 test_loader，不是 val_loader！
final_test_loss_random, final_test_acc_random = validate(
    classifier_random, test_loader, criterion, device)

print(f"\n🔍 随机初始化模型最终测试结果:")
print(f"   - 测试损失 (Loss): {final_test_loss_random:.4f}")
print(f"   - 测试准确率 (Accuracy): {final_test_acc_random:.4f}")


🧪 对照实验：随机初始化编码器（无预训练）
🔍 实验 B（随机+冻结）可训练参数量: 1,427,394 (约 1427.4K)
[Random] Epoch 1/100 | Train Loss: 0.6449, Acc: 0.7240 | Val Loss: 0.5936, Acc: 0.8000
🎉 [Random] New best model cached in memory (Val Acc: 0.8000)
[Random] Epoch 2/100 | Train Loss: 0.5703, Acc: 0.8000 | Val Loss: 0.5392, Acc: 0.8000
[Random] Epoch 3/100 | Train Loss: 0.5260, Acc: 0.8000 | Val Loss: 0.5073, Acc: 0.8000
[Random] Epoch 4/100 | Train Loss: 0.5093, Acc: 0.8000 | Val Loss: 0.4899, Acc: 0.8000
[Random] Epoch 5/100 | Train Loss: 0.4888, Acc: 0.8000 | Val Loss: 0.4809, Acc: 0.8000
[Random] Epoch 6/100 | Train Loss: 0.4828, Acc: 0.8000 | Val Loss: 0.4745, Acc: 0.8000
[Random] Epoch 7/100 | Train Loss: 0.4772, Acc: 0.8000 | Val Loss: 0.4685, Acc: 0.8000
[Random] Epoch 8/100 | Train Loss: 0.4655, Acc: 0.8000 | Val Loss: 0.4625, Acc: 0.8000
[Random] Epoch 9/100 | Train Loss: 0.4576, Acc: 0.8000 | Val Loss: 0.4561, Acc: 0.8000
[Random] Epoch 10/100 | Train Loss: 0.4500, Acc: 0.8000 | Val Loss: 0.4497, Acc: 0.8000
[R

In [ ]:
# ==============================
# 实验 C：纯线性分类器（无编码器）
# ==============================

print("\n" + "=" * 60)
print("🧪 对照实验：纯线性分类器（Linear Only）")
print("=" * 60)

# 获取信号长度 L（从数据中推断）
sample_input, _ = next(iter(train_loader))
L = sample_input.shape[-1]  # 假设 x: [B, 1, L]


class LinearOnlyClassifier(nn.Module):

    def __init__(self, input_len, num_classes):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(input_len, num_classes)

    def forward(self, x):
        # x: [B, 1, L] → [B, L]
        x = self.flatten(x)  # [B, L]
        return self.linear(x)


# 创建模型
linear_model = LinearOnlyClassifier(input_len=L,
                                    num_classes=num_classes).to(device)

# 优化器（线性模型可稍大学习率）
lr_linear = 1e-3
optimizer_linear = torch.optim.AdamW(linear_model.parameters(),
                                     lr=lr_linear,
                                     weight_decay=1e-4)
scheduler_linear = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_linear,
                                                              T_max=num_epochs)

# 训练循环（复用已有函数）
best_val_acc_lin = 0.0
epochs_no_improve_lin = 0
early_stop_lin = False

for epoch in range(num_epochs):
    if early_stop_lin:
        print("🛑 线性模型：触发早停！训练提前结束。")
        break

    train_loss, train_acc = train_epoch(linear_model, train_loader, criterion,
                                        optimizer_linear, device)
    val_loss, val_acc = validate(linear_model, val_loader, criterion, device)
    scheduler_linear.step()

    print(f"[Linear] Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    if val_acc > best_val_acc_lin + EARLY_STOP_DELTA:
        best_val_acc_lin = val_acc
        epochs_no_improve_lin = 0
        torch.save(linear_model.state_dict(), "best_linear_classifier.pth")
        print(f"🎉 [Linear] New best model saved (Val Acc: {val_acc:.4f})")
    else:
        epochs_no_improve_lin += 1
        if epochs_no_improve_lin >= PATIENCE:
            early_stop_lin = True
            print(f"⚠️ [Linear] 连续 {PATIENCE} 轮未提升，早停...")

print(f"\n✅ 线性分类器训练完成！最佳验证准确率: {best_val_acc_lin:.4f}")


🧪 对照实验：纯线性分类器（Linear Only）
[Linear] Epoch 1/100 | Train Loss: 1.7088, Acc: 0.2380 | Val Loss: 1.7282, Acc: 0.2760
🎉 [Linear] New best model saved (Val Acc: 0.2760)
[Linear] Epoch 2/100 | Train Loss: 1.2949, Acc: 0.5640 | Val Loss: 1.8286, Acc: 0.2900
🎉 [Linear] New best model saved (Val Acc: 0.2900)
[Linear] Epoch 3/100 | Train Loss: 1.0975, Acc: 0.6640 | Val Loss: 1.8965, Acc: 0.3240
🎉 [Linear] New best model saved (Val Acc: 0.3240)
[Linear] Epoch 4/100 | Train Loss: 0.9659, Acc: 0.7160 | Val Loss: 1.9382, Acc: 0.3320
🎉 [Linear] New best model saved (Val Acc: 0.3320)
[Linear] Epoch 5/100 | Train Loss: 0.8812, Acc: 0.7520 | Val Loss: 1.9828, Acc: 0.3420
🎉 [Linear] New best model saved (Val Acc: 0.3420)
[Linear] Epoch 6/100 | Train Loss: 0.7983, Acc: 0.7860 | Val Loss: 2.0178, Acc: 0.3500
🎉 [Linear] New best model saved (Val Acc: 0.3500)
[Linear] Epoch 7/100 | Train Loss: 0.7298, Acc: 0.8200 | Val Loss: 2.0559, Acc: 0.3460
[Linear] Epoch 8/100 | Train Loss: 0.6726, Acc: 0.8520 | Val Lo

In [124]:
print("\n" + "=" * 70)
print("📊 最终性能对比")
print("=" * 70)
print(f"{'方法':<25} | {'最佳验证准确率'}")
print("-" * 70)
print(f"{'MAE 预训练 + 微调':<25} | {best_val_acc:.4f}")
print(f"{'随机初始化（无预训练）':<25} | {best_val_acc_rand:.4f}")
print("=" * 70)

if best_val_acc > best_val_acc_rand:
    print("✅ 预训练显著提升了性能！")
else:
    print("⚠️ 预训练未带来提升（可能数据充足或任务简单）")


📊 最终性能对比
方法                        | 最佳验证准确率
----------------------------------------------------------------------
MAE 预训练 + 微调              | 0.9973
随机初始化（无预训练）               | 0.9977
⚠️ 预训练未带来提升（可能数据充足或任务简单）
